# Import Library

In [1]:
import pandas as pd
import pickle
import numpy as np
import os

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.svm import SVC
from sklearn.metrics import classification_report

from imblearn.over_sampling import SMOTE

# Import Data

In [2]:
df = pd.read_csv('maintenance_failure.csv')
df.head()

,Unnamed: 0,UDI,Product ID,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Failure Type
0,0,1,M14860,298.1,308.6,1551,42.8,0,No Failure
1,1,2,L47181,298.2,308.7,1408,46.3,3,No Failure
2,2,3,L47182,298.1,308.5,1498,49.4,5,No Failure
3,3,4,L47183,298.2,308.6,1433,39.5,7,No Failure
4,4,5,L47184,298.2,308.7,1408,40.0,9,No Failure


# Data Exploration & Cleaning

## Exploratory Data Analysis

In [3]:
# Check Missing Values
df.isnull().sum()

Unnamed: 0                 0
UDI                        0
Product ID                 0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Failure Type               0
dtype: int64

In [4]:
# Check Data Types
df.dtypes

Unnamed: 0                   int64
UDI                          int64
Product ID                  object
Air temperature [K]        float64
Process temperature [K]    float64
Rotational speed [rpm]       int64
Torque [Nm]                float64
Tool wear [min]              int64
Failure Type                object
dtype: object

In [5]:
# Check Unique Values
df.nunique()

Unnamed: 0                 10000
UDI                        10000
Product ID                 10000
Air temperature [K]           93
Process temperature [K]       82
Rotational speed [rpm]       941
Torque [Nm]                  577
Tool wear [min]              246
Failure Type                   6
dtype: int64

In [6]:
# Check for multicollinearity
df.drop(columns=['Unnamed: 0', 'UDI', 'Product ID', 'Failure Type'], axis=1).corr()

,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min]
Air temperature [K],1.000000,0.876107,0.022670,-0.013778,0.013853
Process temperature [K],0.876107,1.000000,0.019277,-0.014061,0.013488
Rotational speed [rpm],0.022670,0.019277,1.000000,-0.875027,0.000223
Torque [Nm],-0.013778,-0.014061,-0.875027,1.000000,-0.003093
Tool wear [min],0.013853,0.013488,0.000223,-0.003093,1.000000


## Data Cleaning

- Remove Variables with High Cardinality
- Remove Variables that can cause multicollinearity

### Remove Variables with High Cardinality

In [7]:
new_df = df.drop(columns=['Unnamed: 0', 'UDI', 'Product ID'], axis=1)

### Remove Variables that can cause multicollinearity

In [8]:
new_df = new_df.drop(columns=['Process temperature [K]'], axis=1)

In [9]:
new_df.head()

,Air temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Failure Type
0,298.1,1551,42.8,0,No Failure
1,298.2,1408,46.3,3,No Failure
2,298.1,1498,49.4,5,No Failure
3,298.2,1433,39.5,7,No Failure
4,298.2,1408,40.0,9,No Failure


# Data Preprocessing

## Train Test Split

In [10]:
X, y = new_df.drop(columns=['Failure Type'], axis=1), new_df['Failure Type']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## SMOTE

In [11]:
smote = SMOTE(sampling_strategy='auto', random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

## Feature Scaling

In [12]:
scaler = StandardScaler()

X_train_resampled = scaler.fit_transform(X_train_resampled)
X_test = scaler.transform(X_test)

# Model Building

## Grid Search

In [54]:
hyperparameters = {'C': [0.1, 1, 10], 'kernel': ['poly']}

clf = GridSearchCV(SVC(), hyperparameters, cv=2, verbose=2, scoring='f1_macro')

clf.fit(X_train_resampled, y_train_resampled)

Fitting 2 folds for each of 3 candidates, totalling 6 fits
[CV] END .................................C=0.1, kernel=poly; total time=   8.2s
[CV] END .................................C=0.1, kernel=poly; total time=   7.9s
[CV] END ...................................C=1, kernel=poly; total time=   5.8s
[CV] END ...................................C=1, kernel=poly; total time=   5.4s
[CV] END ..................................C=10, kernel=poly; total time=   5.2s
[CV] END ..................................C=10, kernel=poly; total time=   5.2s


GridSearchCV(cv=2, estimator=SVC(),
             param_grid={'C': [0.1, 1, 10], 'kernel': ['poly']},
             scoring='f1_macro', verbose=2)

In [40]:
clf.best_params_

{'C': 10, 'kernel': 'poly'}

## SVC Model

In [13]:
svc = SVC(C=10, kernel='poly')

svc.fit(X_train_resampled, y_train_resampled)

SVC(C=10, kernel='poly')

In [14]:
y_pred = svc.predict(X_test)

In [15]:
print(classification_report(y_test, y_pred))

                          precision    recall  f1-score   support

Heat Dissipation Failure       0.11      0.87      0.19        15
              No Failure       1.00      0.57      0.73      1935
      Overstrain Failure       0.45      1.00      0.62        13
           Power Failure       0.49      0.95      0.64        20
         Random Failures       0.00      0.17      0.00         6
       Tool Wear Failure       0.08      0.73      0.14        11

                accuracy                           0.58      2000
               macro avg       0.35      0.71      0.39      2000
            weighted avg       0.97      0.58      0.71      2000



# Save Model & Scaler

In [81]:
# Specify the file path where you want to save the Pickle file
file_path = "model_and_scaler.pkl"

# Create a dictionary to store both the model and scaler
data_to_save = {
    "model": svc,
    "scaler": scaler,
}

# Save the data to the Pickle file
with open(file_path, "wb") as file:
    pickle.dump(data_to_save, file)
